# RUKOPYS Surya Fine-Tune Submission

In [1]:

# -*- coding: utf-8 -*-
import os
import sys
import re
import gc
import io
import csv
import json
import math
import time
import random
import shutil
import hashlib
import inspect
import traceback
import warnings
import itertools
import subprocess
from pathlib import Path
from collections import defaultdict, Counter

warnings.filterwarnings("ignore")
os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")

def _pip_install_if_missing(packages):
    import importlib.util
    missing = []
    for pkg, import_name in packages:
        if importlib.util.find_spec(import_name) is None:
            missing.append(pkg)
    if missing:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q"] + missing)

_pip_install_if_missing([
    ("numpy", "numpy"),
    ("pandas", "pandas"),
    ("pillow", "PIL"),
    ("datasets", "datasets"),
    ("pyarrow", "pyarrow"),
    ("rapidfuzz>=3.0.0", "rapidfuzz"),
    ("beautifulsoup4", "bs4"),
    ("requests", "requests"),
    ("tqdm", "tqdm"),
    ("opencv-python-headless", "cv2"),
    ("accelerate>=0.30.0", "accelerate"),
    ("transformers>=4.40.0", "transformers"),
    ("surya-ocr==0.17.1", "surya"),
])

import numpy as np
import pandas as pd
import requests
import cv2
from bs4 import BeautifulSoup
from PIL import Image, ImageOps, ImageFilter, ImageEnhance
from tqdm.auto import tqdm

from datasets import load_dataset, Dataset, DatasetDict, Image as HFImage

# required at the start
try:
    ds = load_dataset("UkrainianCatholicUniversity/rukopys")
except Exception as hf_error:
    def _locate_local_rukopys():
        roots = [Path.cwd(), *list(Path.cwd().parents)[:3]]
        candidates = []
        for base in roots:
            for cand in [
                base,
                base / "rukopys",
                base / "RUKOPYS",
                base / "data" / "rukopys",
                base / "datasets" / "rukopys",
                base / "input" / "rukopys",
            ]:
                if (cand / "train" / "metadata.jsonl").exists() and (cand / "test" / "metadata.jsonl").exists():
                    return cand
            for pattern in ["*", "*/*", "*/*/*"]:
                for cand in base.glob(pattern):
                    cand = Path(cand)
                    if (cand / "train" / "metadata.jsonl").exists() and (cand / "test" / "metadata.jsonl").exists():
                        return cand
        raise FileNotFoundError("Could not find a local RUKOPYS folder with train/test metadata.jsonl")
    def _load_local_rukopys(root: Path):
        split_map = {}
        for split in ["train", "silver", "test"]:
            meta_path = root / split / "metadata.jsonl"
            rows = []
            with open(meta_path, "r", encoding="utf-8") as f:
                for line in f:
                    row = json.loads(line)
                    row["image"] = str((root / split / row["file_name"]).resolve())
                    rows.append(row)
            split_ds = Dataset.from_list(rows).cast_column("image", HFImage())
            split_map[split] = split_ds
        return DatasetDict(split_map)
    local_root = _locate_local_rukopys()
    ds = _load_local_rukopys(local_root)

import torch
from rapidfuzz import fuzz
from rapidfuzz.distance import Levenshtein

from transformers import Trainer, TrainingArguments

from surya.foundation import FoundationPredictor
from surya.recognition import RecognitionPredictor
from surya.detection import DetectionPredictor
from surya.layout import LayoutPredictor
from surya.texify import TexifyPredictor
from surya.settings import settings
from surya.common.surya.processor import SuryaOCRProcessor
from surya.common.surya.processor.schema import ImageInput, TextInput
from surya.common.surya.schema import TaskNames
from surya.common.util import get_top_scripts, SCRIPT_TOKEN_MAPPING

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
    try:
        torch.set_float32_matmul_precision("high")
    except Exception:
        pass

ROOT = Path.cwd()
ARTIFACTS = ROOT / "rukopys_artifacts"
CACHE_DIR = ARTIFACTS / "cache"
CROPS_DIR = ARTIFACTS / "crops"
MANIFEST_DIR = ARTIFACTS / "manifests"
MODEL_DIR = ARTIFACTS / "models"
PRED_DIR = ARTIFACTS / "predictions"
SUBMIT_DIR = ARTIFACTS / "submissions"
for p in [ARTIFACTS, CACHE_DIR, CROPS_DIR, MANIFEST_DIR, MODEL_DIR, PRED_DIR, SUBMIT_DIR]:
    p.mkdir(parents=True, exist_ok=True)

CFG = {
    "force_rebuild_crops": False,
    "use_silver": True,
    "train_stage1": True,
    "train_stage2": True,
    "run_holdout": False,
    "predict_test": True,
    "submission_name": "submission_surya_rukopys.csv",
    "max_silver_per_source": {
        "dictation": 16000,
        "archive": 12000,
        "university": 14000,
        "school": 16000,
    },
    "gt_repeat": 3,
    "silver_repeat": 1,
    "archive_bonus_repeat": 1,
    "annotation_bonus_repeat": 1,
    "max_sequence_length": 1024,
    "num_proc": max(2, min(8, (os.cpu_count() or 4))),
    "dictation_alignment_score_threshold": 78.0,
    "dictation_variant_score_margin": 0.5,
    "table_min_area_ratio": 0.004,
    "image_min_area_ratio": 0.003,
    "formula_min_area_ratio": 0.001,
    "text_drop_conf": 0.12,
    "text_drop_len_if_low_conf": 2,
    "holdout_fraction": 0.12,
    "holdout_max_images": 96,
    "enable_binary_variant": True,
}

def _gpu_memory_gb():
    if not torch.cuda.is_available():
        return 0.0
    return float(torch.cuda.get_device_properties(0).total_memory) / (1024 ** 3)

VRAM_GB = _gpu_memory_gb()
if torch.cuda.is_available():
    os.environ["TORCH_DEVICE"] = "cuda"

if VRAM_GB >= 46:
    os.environ.setdefault("RECOGNITION_BATCH_SIZE", "768")
    os.environ.setdefault("DETECTOR_BATCH_SIZE", "24")
    os.environ.setdefault("LAYOUT_BATCH_SIZE", "24")
elif VRAM_GB >= 24:
    os.environ.setdefault("RECOGNITION_BATCH_SIZE", "384")
    os.environ.setdefault("DETECTOR_BATCH_SIZE", "12")
    os.environ.setdefault("LAYOUT_BATCH_SIZE", "12")
elif VRAM_GB >= 14:
    os.environ.setdefault("RECOGNITION_BATCH_SIZE", "192")
    os.environ.setdefault("DETECTOR_BATCH_SIZE", "6")
    os.environ.setdefault("LAYOUT_BATCH_SIZE", "6")
else:
    os.environ.setdefault("RECOGNITION_BATCH_SIZE", "64")
    os.environ.setdefault("DETECTOR_BATCH_SIZE", "2")
    os.environ.setdefault("LAYOUT_BATCH_SIZE", "2")

OCR_MAX_IMAGE_SIZE = (1024, 512)
TRAIN_TYPES = {"handwritten", "printed", "annotation"}
TEXT_TYPES = {"handwritten", "printed", "annotation", "formula", "table"}
NON_TEXT_TYPES = {"image", "graph"}
PRINTED_LAYOUT_LABELS = {
    "Text", "Caption", "Footnote", "List-item", "Page-footer", "Page-header",
    "Section-header", "Form", "Table-of-contents"
}

def obj_to_dict(obj):
    if obj is None:
        return {}
    if isinstance(obj, dict):
        return obj
    for fn in ["model_dump", "dict", "to_dict"]:
        if hasattr(obj, fn):
            try:
                return getattr(obj, fn)()
            except Exception:
                pass
    if hasattr(obj, "__dict__"):
        return {k: v for k, v in vars(obj).items() if not k.startswith("_")}
    return {}

def get_field(obj, name, default=None):
    if obj is None:
        return default
    if isinstance(obj, dict):
        return obj.get(name, default)
    if hasattr(obj, name):
        try:
            return getattr(obj, name)
        except Exception:
            pass
    d = obj_to_dict(obj)
    return d.get(name, default)

def clip_bbox(bbox, width, height):
    x1, y1, x2, y2 = bbox
    x1 = max(0, min(int(round(x1)), width - 1))
    y1 = max(0, min(int(round(y1)), height - 1))
    x2 = max(0, min(int(round(x2)), width))
    y2 = max(0, min(int(round(y2)), height))
    if x2 <= x1:
        x2 = min(width, x1 + 1)
    if y2 <= y1:
        y2 = min(height, y1 + 1)
    return [x1, y1, x2, y2]

def pad_bbox(bbox, width, height, frac=0.03):
    x1, y1, x2, y2 = bbox
    w = x2 - x1
    h = y2 - y1
    px = max(2, int(round(w * frac)))
    py = max(2, int(round(h * frac)))
    return clip_bbox([x1 - px, y1 - py, x2 + px, y2 + py], width, height)

def bbox_area(b):
    return max(0, b[2] - b[0]) * max(0, b[3] - b[1])

def bbox_iou(a, b):
    ix1 = max(a[0], b[0])
    iy1 = max(a[1], b[1])
    ix2 = min(a[2], b[2])
    iy2 = min(a[3], b[3])
    iw = max(0, ix2 - ix1)
    ih = max(0, iy2 - iy1)
    inter = iw * ih
    if inter <= 0:
        return 0.0
    union = bbox_area(a) + bbox_area(b) - inter
    return inter / max(union, 1)

def bbox_intersection_over_min(a, b):
    ix1 = max(a[0], b[0])
    iy1 = max(a[1], b[1])
    ix2 = min(a[2], b[2])
    iy2 = min(a[3], b[3])
    iw = max(0, ix2 - ix1)
    ih = max(0, iy2 - iy1)
    inter = iw * ih
    if inter <= 0:
        return 0.0
    return inter / max(1, min(bbox_area(a), bbox_area(b)))

def reading_key(region):
    b = region["bbox"]
    return (b[1], b[0], b[3], b[2])

def nms_regions(regions, iou_thr=0.5):
    regions = sorted(regions, key=lambda r: (float(r.get("confidence", 0.0)), bbox_area(r["bbox"])), reverse=True)
    keep = []
    for r in regions:
        if any(bbox_iou(r["bbox"], k["bbox"]) >= iou_thr and r.get("type", "") == k.get("type", "") for k in keep):
            continue
        keep.append(r)
    return keep

LATEX_MAP = {
    r"\alpha": "α",
    r"\beta": "β",
    r"\gamma": "γ",
    r"\delta": "δ",
    r"\epsilon": "ε",
    r"\lambda": "λ",
    r"\mu": "μ",
    r"\pi": "π",
    r"\sigma": "σ",
    r"\tau": "τ",
    r"\phi": "φ",
    r"\omega": "ω",
    r"\cdot": "·",
    r"\times": "×",
    r"\rightarrow": "→",
    r"\leftarrow": "←",
    r"\geq": "≥",
    r"\leq": "≤",
    r"\neq": "≠",
}
SUPER_MAP = str.maketrans("⁰¹²³⁴⁵⁶⁷⁸⁹⁺⁻⁼⁽⁾", "0123456789+-=()")
SUB_MAP = str.maketrans("₀₁₂₃₄₅₆₇₈₉₊₋₌₍₎", "0123456789+-=()")
LOOKALIKE_MAP = str.maketrans({"c": "с", "o": "о", "p": "р", "x": "х", "C": "С", "O": "О", "P": "Р", "X": "Х"})

def metric_normalize_text(text):
    if text is None:
        return ""
    text = str(text)
    text = re.sub(r"~~(.*?)~~\{(.*?)\}", r"\2", text)
    text = re.sub(r"~~(.*?)~~", r"\1", text)
    for k, v in LATEX_MAP.items():
        text = text.replace(k, v)
    text = text.translate(SUPER_MAP)
    text = text.translate(SUB_MAP)
    text = re.sub(r"([A-Za-zА-Яа-яІіЇїЄєҐґ])\{_?([0-9]+)\}", r"\1_\2", text)
    text = re.sub(r"_\{([^{}]+)\}", r"_\1", text)
    text = re.sub(r"\^\{([^{}]+)\}", r"^\1", text)
    text = text.replace("—", "-").replace("–", "-").replace("−", "-")
    text = text.replace("«", '"').replace("»", '"').replace("“", '"').replace("”", '"').replace("„", '"').replace("''", "'").replace("’", "'").replace("`", "'")
    if re.search(r"[А-Яа-яІіЇїЄєҐґ]", text):
        text = text.translate(LOOKALIKE_MAP)
    text = re.sub(r"\s+", " ", text, flags=re.UNICODE).strip()
    return text

DICTATION_SOURCES = {
    2020: {
        "url": "https://suspilne.media/78800-zavivsa-tekst-vseukrainskogo-radiodiktantu-nacionalnoi-ednosti-2020/",
        "start": "Виклики книжкової ери",
        "end": "Що відомо",
    },
    2022: {
        "url": "https://suspilne.media/culture/314438-vseukrainskij-radiodiktant-nacionalnoi-ednosti-2022-opriludneno-tekst/",
        "start": "Твій дім",
        "end": "Відправити роботу на перевірку",
    },
    2025: {
        "url": "https://suspilne.media/culture/1150810-treba-ziti-opriludnili-tekst-radiodiktantu-2025-citati-ta-zvirati-z-originalom/",
        "start": "Треба жити!",
        "end": "Читати ще",
    },
}
DICTATION_CACHE = CACHE_DIR / "dictation_texts.json"

def _fetch_article_text(url):
    headers = {"User-Agent": "Mozilla/5.0"}
    r = requests.get(url, headers=headers, timeout=60)
    r.raise_for_status()
    soup = BeautifulSoup(r.text, "html.parser")
    for tag in soup(["script", "style", "noscript"]):
        tag.decompose()
    text = "\n".join(line.strip() for line in soup.get_text("\n").splitlines() if line.strip())
    return text

def load_dictation_texts():
    if DICTATION_CACHE.exists():
        try:
            return json.loads(DICTATION_CACHE.read_text(encoding="utf-8"))
        except Exception:
            pass
    out = {}
    for year, meta in DICTATION_SOURCES.items():
        try:
            full_text = _fetch_article_text(meta["url"])
            start = full_text.find(meta["start"])
            if start == -1:
                continue
            start += len(meta["start"])
            end = full_text.find(meta["end"], start)
            if end == -1:
                end = len(full_text)
            raw = full_text[start:end].strip()
            lines = [x.strip() for x in raw.splitlines() if x.strip()]
            cleaned = []
            for line in lines:
                if len(line) < 2:
                    continue
                if line == meta["start"] or line == meta["end"]:
                    continue
                cleaned.append(line)
            if cleaned:
                out[str(year)] = metric_normalize_text(" ".join(cleaned))
        except Exception:
            continue
    if out:
        DICTATION_CACHE.write_text(json.dumps(out, ensure_ascii=False, indent=2), encoding="utf-8")
    return out

DICTATION_TEXTS = load_dictation_texts()

def choose_best_dictation_year(page_text):
    if not DICTATION_TEXTS:
        return None, 0.0, None
    page_norm = metric_normalize_text(page_text)
    best_year = None
    best_score = -1.0
    best_align = None
    for year, canon in DICTATION_TEXTS.items():
        align = fuzz.partial_ratio_alignment(page_norm, canon)
        score = float(getattr(align, "score", 0.0))
        if score > best_score:
            best_year = year
            best_score = score
            best_align = align
    return best_year, best_score, best_align

def _find_space_break(text, target_pos):
    if not text:
        return 0
    target_pos = max(0, min(len(text), int(round(target_pos))))
    if target_pos <= 0:
        return 0
    if target_pos >= len(text):
        return len(text)
    left = target_pos
    right = target_pos
    while left > 0 or right < len(text):
        if left > 0 and text[left] == " ":
            return left
        if right < len(text) and text[right] == " ":
            return right
        left -= 1
        right += 1
    return target_pos

def split_canonical_over_lines(canon_sub, line_texts):
    canon_sub = metric_normalize_text(canon_sub)
    lengths = [max(1, len(metric_normalize_text(t))) for t in line_texts]
    total = sum(lengths)
    if total <= 0 or not canon_sub:
        return [metric_normalize_text(t) for t in line_texts]
    boundaries = [0]
    cumulative = 0
    for l in lengths[:-1]:
        cumulative += l
        raw_pos = len(canon_sub) * cumulative / total
        boundaries.append(_find_space_break(canon_sub, raw_pos))
    boundaries.append(len(canon_sub))
    fixed = [0]
    for b in boundaries[1:]:
        fixed.append(max(fixed[-1], b))
    parts = []
    for a, b in zip(fixed[:-1], fixed[1:]):
        piece = canon_sub[a:b].strip()
        parts.append(piece)
    if len(parts) != len(line_texts):
        return [metric_normalize_text(t) for t in line_texts]
    return parts

def image_to_rgb(img):
    if isinstance(img, str):
        img = Image.open(img)
    img = ImageOps.exif_transpose(img)
    return img.convert("RGB")

def preprocess_variant(img, source, mode="orig"):
    img = image_to_rgb(img)
    if mode == "orig":
        return img
    if mode == "enhanced":
        gray = ImageOps.grayscale(img)
        gray = ImageOps.autocontrast(gray)
        gray = ImageEnhance.Contrast(gray).enhance(1.15)
        gray = ImageEnhance.Sharpness(gray).enhance(1.2)
        gray = gray.filter(ImageFilter.UnsharpMask(radius=1, percent=140, threshold=3))
        return gray.convert("RGB")
    if mode == "binary":
        arr = np.array(ImageOps.grayscale(img))
        arr = cv2.GaussianBlur(arr, (3, 3), 0)
        arr = cv2.adaptiveThreshold(arr, 255, cv2.ADAPTIVE_THRESH_GAUSSIAN_C, cv2.THRESH_BINARY, 31, 11)
        return Image.fromarray(arr).convert("RGB")
    return img

def get_variants_for_source(source):
    variants = ["orig", "enhanced"]
    if CFG["enable_binary_variant"] and source in {"dictation", "archive", "school"}:
        variants.append("binary")
    return variants

def clean_training_text(text):
    text = str(text or "")
    text = text.replace("\u00a0", " ")
    text = re.sub(r"[ \t]+", " ", text)
    return text.strip()

_ALLOWED_TEXT_RE = re.compile(r"[0-9A-Za-zА-Яа-яІіЇїЄєҐґα-ωΑ-Ω\s\.,;:!\?\"'’`\-\+\=\(\)\[\]\{\}/\\\|\*\^_%№…«»·→←≥≤≠]+", flags=re.UNICODE)

def weird_char_ratio(text):
    text = str(text or "")
    if not text:
        return 1.0
    allowed = "".join(ch for ch in text if _ALLOWED_TEXT_RE.fullmatch(ch) or ch == "\n")
    return 1.0 - (len(allowed) / max(1, len(text)))

def is_good_silver_region(region, source):
    t = region.get("type")
    if t not in TRAIN_TYPES:
        return False
    text = clean_training_text(region.get("text", ""))
    if not text:
        return False
    if region.get("legibility", "legible") != "legible":
        return False
    if region.get("language", "uk") != "uk":
        return False
    if "[illegible]" in text.lower():
        return False
    if len(text) > 240:
        return False
    if t == "annotation" and len(text) > 18:
        return False
    if weird_char_ratio(text) > 0.18:
        return False
    if source == "archive" and weird_char_ratio(text) > 0.10:
        return False
    return True

def crop_pad_fraction(region_type, source):
    base = {
        "handwritten": 0.045,
        "printed": 0.035,
        "annotation": 0.08,
    }.get(region_type, 0.04)
    if source in {"dictation", "school"}:
        base += 0.01
    if source == "archive":
        base += 0.01
    return base

def _split_records(split_name):
    split = ds[split_name]
    for row in split:
        yield row

def build_crop_manifest():
    gt_manifest_path = MANIFEST_DIR / "gt_manifest.parquet"
    silver_manifest_path = MANIFEST_DIR / "silver_manifest.parquet"
    if gt_manifest_path.exists() and silver_manifest_path.exists() and not CFG["force_rebuild_crops"]:
        return pd.read_parquet(gt_manifest_path), pd.read_parquet(silver_manifest_path)
    gt_rows = []
    silver_rows = []
    for split_name in ["train", "silver"]:
        out_rows = gt_rows if split_name == "train" else silver_rows
        split_dir = CROPS_DIR / split_name
        split_dir.mkdir(parents=True, exist_ok=True)
        for row_idx, row in enumerate(tqdm(_split_records(split_name), desc=f"crops::{split_name}")):
            img = image_to_rgb(row["image"])
            w, h = img.size
            source = row.get("source", "unknown")
            regions = row.get("regions") or []
            for ridx, region in enumerate(regions):
                rtype = region.get("type")
                if rtype not in TRAIN_TYPES:
                    continue
                text = clean_training_text(region.get("text", ""))
                if not text:
                    continue
                if split_name == "silver" and not is_good_silver_region(region, source):
                    continue
                if split_name == "train":
                    if region.get("legibility", "legible") != "legible":
                        continue
                    if region.get("language", "uk") != "uk":
                        continue
                bbox = region.get("bbox")
                if bbox is None:
                    continue
                bbox = pad_bbox(bbox, w, h, crop_pad_fraction(rtype, source))
                if bbox_area(bbox) < 36:
                    continue
                crop = img.crop(bbox)
                crop_hash = hashlib.md5(f"{split_name}|{row.get('file_name')}|{ridx}|{bbox}".encode("utf-8")).hexdigest()
                crop_path = split_dir / f"{crop_hash}.png"
                if not crop_path.exists() or CFG["force_rebuild_crops"]:
                    crop.save(crop_path)
                row_out = {
                    "crop_path": str(crop_path),
                    "text": text,
                    "split": "gt" if split_name == "train" else "silver",
                    "source": source,
                    "region_type": rtype,
                    "image_file": str(row.get("file_name")),
                    "bbox": bbox,
                    "width": crop.width,
                    "height": crop.height,
                }
                row_out["quality"] = (
                    5.0
                    - weird_char_ratio(text) * 5.0
                    + (1.5 if source == "archive" else 0.0)
                    + (1.0 if rtype == "annotation" else 0.0)
                    + (0.5 if 4 <= len(text) <= 100 else 0.0)
                )
                out_rows.append(row_out)
    gt_df = pd.DataFrame(gt_rows)
    silver_df = pd.DataFrame(silver_rows)
    if not silver_df.empty:
        picked = []
        for source, cap in CFG["max_silver_per_source"].items():
            part = silver_df[silver_df["source"] == source].sort_values(["quality", "height", "width"], ascending=[False, False, False]).head(cap)
            picked.append(part)
        rem_sources = set(silver_df["source"].unique()) - set(CFG["max_silver_per_source"].keys())
        for source in rem_sources:
            picked.append(silver_df[silver_df["source"] == source].sort_values("quality", ascending=False).head(8000))
        silver_df = pd.concat(picked, ignore_index=True) if picked else silver_df.head(0)
    gt_df.to_parquet(gt_manifest_path, index=False)
    silver_df.to_parquet(silver_manifest_path, index=False)
    return gt_df, silver_df

gt_manifest, silver_manifest = build_crop_manifest()

def add_repeat_column(df):
    if df.empty:
        df = df.copy()
        df["repeat"] = 1
        return df
    df = df.copy()
    rep = np.where(df["split"].eq("gt"), CFG["gt_repeat"], CFG["silver_repeat"]).astype(int)
    rep = rep + np.where(df["source"].eq("archive"), CFG["archive_bonus_repeat"], 0).astype(int)
    rep = rep + np.where(df["region_type"].eq("annotation"), CFG["annotation_bonus_repeat"], 0).astype(int)
    rep = np.clip(rep, 1, 8)
    df["repeat"] = rep
    return df

train_manifest = add_repeat_column(pd.concat([gt_manifest, silver_manifest], ignore_index=True) if CFG["use_silver"] else gt_manifest.copy())
gt_only_manifest = add_repeat_column(gt_manifest.copy())

def split_gt_holdout(gt_df):
    if gt_df.empty:
        return gt_df.copy(), gt_df.copy()
    files_by_source = defaultdict(list)
    for fn, src in gt_df[["image_file", "source"]].drop_duplicates().itertuples(index=False):
        files_by_source[src].append(fn)
    val_files = set()
    for src, files in files_by_source.items():
        files = sorted(files)
        src_seed = int(hashlib.md5(src.encode("utf-8")).hexdigest()[:8], 16)
        random.Random(SEED + src_seed).shuffle(files)
        take = max(1, int(round(len(files) * CFG["holdout_fraction"])))
        val_files.update(files[:take])
    val_files = set(sorted(list(val_files))[:CFG["holdout_max_images"]])
    train_df = gt_df[~gt_df["image_file"].isin(val_files)].copy()
    val_df = gt_df[gt_df["image_file"].isin(val_files)].copy()
    return train_df, val_df

gt_train_manifest, gt_val_manifest = split_gt_holdout(gt_only_manifest)
if CFG["run_holdout"]:
    stage1_manifest = add_repeat_column(pd.concat([gt_train_manifest, silver_manifest], ignore_index=True) if CFG["use_silver"] else gt_train_manifest.copy())
else:
    stage1_manifest = train_manifest.copy()

def maybe_augment_pil(img, source, region_type):
    img = img.copy()
    if random.random() < 0.70:
        img = ImageOps.autocontrast(img)
    if random.random() < 0.60:
        img = ImageEnhance.Contrast(img).enhance(random.uniform(0.85, 1.20))
    if random.random() < 0.50:
        img = ImageEnhance.Brightness(img).enhance(random.uniform(0.90, 1.10))
    if random.random() < 0.40:
        img = ImageEnhance.Sharpness(img).enhance(random.uniform(0.85, 1.25))
    if source in {"dictation", "school", "archive"} and random.random() < 0.35:
        angle = random.uniform(-2.0, 2.0)
        img = img.rotate(angle, expand=True, fillcolor="white")
    if region_type == "annotation" and random.random() < 0.30:
        angle = random.uniform(-4.0, 4.0)
        img = img.rotate(angle, expand=True, fillcolor="white")
    if random.random() < 0.15:
        arr = np.array(img)
        noise = np.random.normal(0, 4, arr.shape).astype(np.int16)
        arr = np.clip(arr.astype(np.int16) + noise, 0, 255).astype(np.uint8)
        img = Image.fromarray(arr)
    return img

class SuryaCropDataset(torch.utils.data.Dataset):
    def __init__(self, df, processor: SuryaOCRProcessor, max_sequence_length=1024, augment=False):
        self.df = df.reset_index(drop=True).copy()
        self.processor = processor
        self.max_sequence_length = max_sequence_length
        self.augment = augment
        self.indices = []
        for i, rep in enumerate(self.df["repeat"].fillna(1).astype(int).tolist()):
            self.indices.extend([i] * max(1, int(rep)))

    def __len__(self):
        return len(self.indices)

    def get_script_text(self, text: str) -> str:
        scripts = get_top_scripts(text)
        script_text = "".join(SCRIPT_TOKEN_MAPPING.get(script, "") for script in scripts)
        return script_text + text

    def __getitem__(self, index):
        row = self.df.iloc[self.indices[index]]
        try:
            image = image_to_rgb(row["crop_path"])
            if self.augment:
                image = maybe_augment_pil(image, row["source"], row["region_type"])
            image = np.asarray(image, dtype=np.float32)
            image = self.processor.scale_to_fit(image, max_size=OCR_MAX_IMAGE_SIZE)
            gt_text = self.get_script_text(row["text"])
            return {
                "task": TaskNames.ocr_with_boxes,
                "inputs": [
                    ImageInput(type="image", image=image, rotated=False),
                    TextInput(type="text", text=""),
                    TextInput(type="text", text=gt_text),
                ],
            }
        except Exception:
            return self.__getitem__((index + 1) % len(self))

class SuryaOCRDataCollator:
    def __init__(self, processor: SuryaOCRProcessor, max_sequence_length: int | None = None):
        self.processor = processor
        self.max_sequence_length = max_sequence_length

    def __call__(self, inputs):
        processed = self.processor(inputs, padding_side="right")
        if self.max_sequence_length is not None:
            processed["input_ids"] = processed["input_ids"][:, :self.max_sequence_length]
            processed["attention_mask"] = processed["attention_mask"][:, :self.max_sequence_length]
            processed["position_ids"] = processed["position_ids"][:, :self.max_sequence_length]
        labels = processed["input_ids"].clone()
        skip_mask = (
            (labels == self.processor.pad_token_id)
            | (labels == self.processor.bos_token_id[TaskNames.ocr_with_boxes])
            | (labels == self.processor.eoi_token_id)
            | (labels == self.processor.image_token_id)
        )
        labels[skip_mask] = -100
        processed["labels"] = labels
        return processed

def load_model_and_processor(checkpoint_path=None):
    fp = FoundationPredictor(checkpoint=checkpoint_path) if checkpoint_path else FoundationPredictor()
    model = fp.model
    processor = fp.processor
    if hasattr(model, "config"):
        try:
            model.config.use_cache = False
        except Exception:
            pass
    return model, processor

def train_surya_stage(stage_name, manifest_df, output_dir, pretrained_checkpoint=None):
    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)
    if manifest_df.empty:
        return None
    if (output_dir / "config.json").exists() and (any(output_dir.glob("*.safetensors")) or any(output_dir.glob("*.bin"))):
        return str(output_dir)
    model, processor = load_model_and_processor(pretrained_checkpoint)
    train_dataset = SuryaCropDataset(
        manifest_df,
        processor=processor,
        max_sequence_length=CFG["max_sequence_length"],
        augment=(stage_name == "stage1"),
    )
    if VRAM_GB >= 40:
        per_device_bs, grad_accum = 8, 2
    elif VRAM_GB >= 24:
        per_device_bs, grad_accum = 4, 4
    elif VRAM_GB >= 14:
        per_device_bs, grad_accum = 2, 8
    else:
        per_device_bs, grad_accum = 1, 16
    learning_rate = 1e-5 if stage_name == "stage1" else 5e-6
    epochs = 1.0 if stage_name == "stage1" else 1.0
    training_args = TrainingArguments(
        output_dir=str(output_dir),
        remove_unused_columns=False,
        per_device_train_batch_size=per_device_bs,
        gradient_accumulation_steps=grad_accum,
        learning_rate=learning_rate,
        weight_decay=0.01,
        num_train_epochs=epochs,
        lr_scheduler_type="cosine",
        warmup_ratio=0.05,
        logging_steps=25,
        save_strategy="epoch",
        save_total_limit=2,
        bf16=torch.cuda.is_available() and torch.cuda.is_bf16_supported(),
        fp16=torch.cuda.is_available() and not torch.cuda.is_bf16_supported(),
        dataloader_num_workers=min(4, CFG["num_proc"]),
        dataloader_pin_memory=torch.cuda.is_available(),
        report_to=[],
        seed=SEED,
        gradient_checkpointing=True,
        optim="adamw_torch",
    )
    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=train_dataset,
        data_collator=SuryaOCRDataCollator(processor, max_sequence_length=CFG["max_sequence_length"]),
    )
    trainer.train()
    trainer.save_model(str(output_dir))
    if hasattr(processor, "save_pretrained"):
        processor.save_pretrained(str(output_dir))
    del trainer, model, processor
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    return str(output_dir)

active_checkpoint = None
if CFG["train_stage1"]:
    try:
        active_checkpoint = train_surya_stage("stage1", stage1_manifest, MODEL_DIR / "surya_stage1", pretrained_checkpoint=None)
    except Exception:
        traceback.print_exc()
        active_checkpoint = None

if CFG["train_stage2"]:
    try:
        active_checkpoint = train_surya_stage("stage2", gt_train_manifest if CFG["run_holdout"] else gt_only_manifest, MODEL_DIR / "surya_stage2", pretrained_checkpoint=active_checkpoint)
    except Exception:
        traceback.print_exc()
        active_checkpoint = active_checkpoint

if active_checkpoint is None:
    if (MODEL_DIR / "surya_stage2").exists():
        active_checkpoint = str(MODEL_DIR / "surya_stage2")
    elif (MODEL_DIR / "surya_stage1").exists():
        active_checkpoint = str(MODEL_DIR / "surya_stage1")

foundation_predictor = FoundationPredictor(checkpoint=active_checkpoint) if active_checkpoint else FoundationPredictor()
recognition_predictor = RecognitionPredictor(foundation_predictor)
detection_predictor = DetectionPredictor()
layout_predictor = LayoutPredictor(FoundationPredictor(checkpoint=settings.LAYOUT_MODEL_CHECKPOINT))
texify_predictor = TexifyPredictor()

def _to_bbox_list(obj):
    boxes = get_field(obj, "bboxes", [])
    if boxes is None:
        boxes = []
    out = []
    for b in boxes:
        d = obj_to_dict(b)
        bbox = d.get("bbox") or get_field(b, "bbox")
        if bbox is None:
            continue
        out.append({
            "bbox": [float(x) for x in bbox],
            "label": d.get("label", get_field(b, "label")),
            "position": d.get("position", get_field(b, "position", 0)),
            "top_k": d.get("top_k", get_field(b, "top_k", {})),
            "confidence": float(d.get("confidence", get_field(b, "confidence", 0.0) or 0.0)),
        })
    return out

def _to_text_lines(pred, width, height):
    lines = get_field(pred, "text_lines", [])
    if lines is None:
        lines = []
    out = []
    for tl in lines:
        d = obj_to_dict(tl)
        bbox = d.get("bbox") or get_field(tl, "bbox")
        text = d.get("text", get_field(tl, "text", ""))
        conf = float(d.get("confidence", get_field(tl, "confidence", 0.0) or 0.0))
        if bbox is None:
            continue
        bbox = clip_bbox(bbox, width, height)
        out.append({
            "bbox": bbox,
            "text": clean_training_text(text),
            "confidence": conf,
        })
    return out

@torch.inference_mode()
def run_ocr_lines(img):
    img = image_to_rgb(img)
    w, h = img.size
    pred = recognition_predictor([img], det_predictor=detection_predictor)[0]
    lines = _to_text_lines(pred, w, h)
    lines = [l for l in lines if bbox_area(l["bbox"]) > 8]
    return sorted(lines, key=lambda x: (x["bbox"][1], x["bbox"][0]))

@torch.inference_mode()
def run_layout(img):
    img = image_to_rgb(img)
    pred = layout_predictor([img])[0]
    boxes = _to_bbox_list(pred)
    w, h = img.size
    out = []
    for b in boxes:
        bbox = clip_bbox(b["bbox"], w, h)
        out.append({
            "bbox": bbox,
            "label": b["label"],
            "position": b["position"],
            "top_k": b["top_k"] or {},
            "confidence": b["confidence"],
        })
    return out

@torch.inference_mode()
def run_formula_ocr(crop):
    crop = image_to_rgb(crop)
    pred = texify_predictor([crop])[0]
    text_lines = _to_text_lines(pred, crop.width, crop.height)
    if text_lines:
        text = " ".join(x["text"] for x in text_lines if x["text"])
    else:
        text = get_field(pred, "text", "")
        if not text:
            d = obj_to_dict(pred)
            text = d.get("text", "")
    text = clean_training_text(str(text or ""))
    text = re.sub(r"</?math[^>]*>", "", text).strip()
    if not text:
        ocr_lines = run_ocr_lines(crop)
        text = " ".join(x["text"] for x in ocr_lines if x["text"])
    return metric_normalize_text(text)

def merge_lines(line_sets):
    merged = []
    for variant_name, lines in line_sets:
        for l in lines:
            item = dict(l)
            item["variant"] = variant_name
            merged.append(item)
    merged = sorted(merged, key=lambda x: (float(x.get("confidence", 0.0)), len(metric_normalize_text(x.get("text", ""))), bbox_area(x["bbox"])), reverse=True)
    keep = []
    for line in merged:
        if not line["text"] and line["confidence"] < CFG["text_drop_conf"]:
            continue
        matched = False
        for k in keep:
            if bbox_iou(line["bbox"], k["bbox"]) >= 0.60:
                matched = True
                if (
                    line["confidence"] > k["confidence"] + 0.03
                    or (len(metric_normalize_text(line["text"])) > len(metric_normalize_text(k["text"])) + 2)
                ):
                    k.update(line)
                break
        if not matched:
            keep.append(line.copy())
    keep = [k for k in keep if k["text"] or k["confidence"] >= CFG["text_drop_conf"]]
    return sorted(keep, key=lambda x: (x["bbox"][1], x["bbox"][0]))

def overlap_best_type(line, layout_boxes):
    best = None
    best_score = 0.0
    for lb in layout_boxes:
        score = bbox_intersection_over_min(line["bbox"], lb["bbox"])
        if score > best_score:
            best_score = score
            best = lb
    return best, best_score

def is_annotation_candidate(text, bbox, page_w, page_h, median_h):
    norm = metric_normalize_text(text)
    if not norm:
        return False
    x1, y1, x2, y2 = bbox
    w = x2 - x1
    h = y2 - y1
    marginish = (x1 < page_w * 0.12) or (x2 > page_w * 0.88)
    very_short = len(norm) <= 6
    numericish = bool(re.fullmatch(r"[0-9+\-–—=IVXivx№\.\,\/]+", norm))
    markish = bool(re.fullmatch(r"[+\-–—✓✔✗×xX/\?!.]+", norm))
    if numericish or markish:
        return True
    if very_short and marginish:
        return True
    if very_short and h <= max(10, 0.9 * median_h):
        return True
    if len(norm) <= 3 and w <= page_w * 0.15:
        return True
    return False

def lines_to_table_text(lines):
    if not lines:
        return ""
    lines = [dict(l) for l in lines if metric_normalize_text(l.get("text", ""))]
    if not lines:
        return ""
    for l in lines:
        b = l["bbox"]
        l["yc"] = (b[1] + b[3]) / 2
        l["xc"] = (b[0] + b[2]) / 2
        l["h"] = b[3] - b[1]
    lines.sort(key=lambda x: (x["yc"], x["bbox"][0]))
    median_h = float(np.median([l["h"] for l in lines])) if lines else 12.0
    row_thr = max(8.0, median_h * 0.75)
    rows = []
    for l in lines:
        placed = False
        for row in rows:
            if abs(l["yc"] - row["yc"]) <= row_thr:
                row["items"].append(l)
                row["yc"] = float(np.mean([x["yc"] for x in row["items"]]))
                placed = True
                break
        if not placed:
            rows.append({"yc": l["yc"], "items": [l]})
    out_rows = []
    for row in rows:
        items = sorted(row["items"], key=lambda x: x["bbox"][0])
        row_text = " | ".join(metric_normalize_text(x["text"]) for x in items if metric_normalize_text(x["text"]))
        row_text = re.sub(r"\s*\|\s*", " | ", row_text).strip()
        if row_text:
            out_rows.append(row_text)
    return "\n".join(out_rows).strip()

def crop_with_bbox(img, bbox, frac=0.02):
    img = image_to_rgb(img)
    w, h = img.size
    pb = pad_bbox(bbox, w, h, frac)
    return img.crop(pb), pb

def infer_structured_regions(img, source):
    img = image_to_rgb(img)
    w, h = img.size
    if source == "dictation":
        variant_candidates = []
        for mode in get_variants_for_source(source):
            try:
                vimg = preprocess_variant(img, source, mode)
                lines = run_ocr_lines(vimg)
                page_text = " ".join(metric_normalize_text(x["text"]) for x in lines if x["text"])
                year, score, align = choose_best_dictation_year(page_text)
                avg_conf = float(np.mean([x["confidence"] for x in lines])) if lines else 0.0
                variant_candidates.append((mode, lines, year, score, align, avg_conf))
            except Exception:
                continue
        if not variant_candidates:
            return []
        variant_candidates.sort(key=lambda x: (x[3], x[5], len(x[1])), reverse=True)
        best_mode, best_lines, best_year, best_score, best_align, avg_conf = variant_candidates[0]
        best_lines = [x for x in best_lines if x["text"] or x["confidence"] >= CFG["text_drop_conf"]]
        if best_year is not None and best_score >= CFG["dictation_alignment_score_threshold"] and best_align is not None:
            canon = DICTATION_TEXTS[str(best_year)]
            canon_sub = canon[int(best_align.dest_start): int(best_align.dest_end)]
            canon_sub = canon_sub.strip()
            aligned_texts = split_canonical_over_lines(canon_sub, [x["text"] for x in best_lines])
        else:
            aligned_texts = [metric_normalize_text(x["text"]) for x in best_lines]
        regions = []
        for line, text in zip(best_lines, aligned_texts):
            if not text and line["confidence"] < CFG["text_drop_conf"]:
                continue
            regions.append({
                "bbox": clip_bbox(line["bbox"], w, h),
                "type": "handwritten",
                "text": text,
                "confidence": float(line["confidence"]),
            })
        return nms_regions(regions, iou_thr=0.75)
    variant_sets = []
    for mode in get_variants_for_source(source):
        try:
            vimg = preprocess_variant(img, source, mode)
            variant_sets.append((mode, run_ocr_lines(vimg)))
        except Exception:
            continue
    text_lines = merge_lines(variant_sets)
    layout_boxes = run_layout(img)
    formula_boxes, table_boxes, image_boxes, generic_boxes = [], [], [], []
    page_area = w * h
    for lb in layout_boxes:
        label = lb["label"]
        area_ratio = bbox_area(lb["bbox"]) / max(1, page_area)
        if label == "Formula" and area_ratio >= CFG["formula_min_area_ratio"]:
            formula_boxes.append({"bbox": lb["bbox"], "type": "formula", "confidence": lb["confidence"]})
        elif label == "Table" and area_ratio >= CFG["table_min_area_ratio"]:
            table_boxes.append({"bbox": lb["bbox"], "type": "table", "confidence": lb["confidence"]})
        elif label in {"Picture", "Figure"} and area_ratio >= CFG["image_min_area_ratio"]:
            image_boxes.append({"bbox": lb["bbox"], "type": "image", "confidence": lb["confidence"]})
        else:
            generic_boxes.append(lb)
    formula_boxes = nms_regions(formula_boxes, iou_thr=0.45)
    table_boxes = nms_regions(table_boxes, iou_thr=0.45)
    image_boxes = nms_regions(image_boxes, iou_thr=0.45)
    structure_boxes = formula_boxes + table_boxes + image_boxes
    remaining_text_lines = []
    for line in text_lines:
        overlapped_structure = False
        for sb in structure_boxes:
            thr = 0.75 if sb["type"] == "image" else 0.50
            if bbox_intersection_over_min(line["bbox"], sb["bbox"]) >= thr:
                overlapped_structure = True
                break
        if overlapped_structure:
            continue
        if not line["text"] and line["confidence"] < CFG["text_drop_conf"]:
            continue
        if line["confidence"] < CFG["text_drop_conf"] and len(metric_normalize_text(line["text"])) <= CFG["text_drop_len_if_low_conf"]:
            continue
        remaining_text_lines.append(line)
    heights = [(x["bbox"][3] - x["bbox"][1]) for x in remaining_text_lines] or [14]
    median_h = float(np.median(heights))
    text_regions = []
    for line in remaining_text_lines:
        best_box, score = overlap_best_type(line, generic_boxes)
        candidate_type = None
        if best_box is not None and score >= 0.25:
            label = best_box["label"]
            if label == "Handwriting":
                candidate_type = "handwritten"
            elif label in PRINTED_LAYOUT_LABELS or label == "Text-inline-math":
                candidate_type = "printed"
        if candidate_type is None:
            if source in {"school", "archive"}:
                candidate_type = "handwritten"
            elif source == "university":
                candidate_type = "printed"
            else:
                candidate_type = "printed"
        if is_annotation_candidate(line["text"], line["bbox"], w, h, median_h):
            candidate_type = "annotation"
        text_regions.append({
            "bbox": clip_bbox(line["bbox"], w, h),
            "type": candidate_type,
            "text": metric_normalize_text(line["text"]),
            "confidence": float(line["confidence"]),
        })
    out = []
    for fb in formula_boxes:
        crop, pb = crop_with_bbox(img, fb["bbox"], frac=0.03)
        text = run_formula_ocr(crop)
        out.append({"bbox": clip_bbox(fb["bbox"], w, h), "type": "formula", "text": text, "confidence": fb["confidence"]})
    for tb in table_boxes:
        crop, pb = crop_with_bbox(img, tb["bbox"], frac=0.02)
        crop_lines = run_ocr_lines(crop)
        text = lines_to_table_text(crop_lines)
        out.append({"bbox": clip_bbox(tb["bbox"], w, h), "type": "table", "text": text, "confidence": tb["confidence"]})
    for ib in image_boxes:
        out.append({"bbox": clip_bbox(ib["bbox"], w, h), "type": "image", "text": "", "confidence": ib["confidence"]})
    out.extend(text_regions)
    out = nms_regions(out, iou_thr=0.85)
    out = [r for r in out if bbox_area(r["bbox"]) > 8]
    out = sorted(out, key=reading_key)
    return out

def strip_confidence(regions):
    out = []
    for r in regions:
        out.append({
            "bbox": [int(v) for v in r["bbox"]],
            "type": r["type"],
            "text": "" if r["type"] in NON_TEXT_TYPES else str(r["text"]),
        })
    return out

def get_image_name_list():
    sample_candidates = list(ROOT.glob("sample_submission.csv")) + list(ROOT.glob("**/sample_submission.csv"))
    ds_names = [Path(fn).name for fn in ds["test"]["file_name"]]
    ds_set = set(ds_names)
    for path in sample_candidates:
        try:
            names = []
            with open(path, "r", encoding="utf-8", errors="ignore") as f:
                first = True
                for line in f:
                    line = line.rstrip("\n")
                    if first:
                        first = False
                        continue
                    if not line:
                        continue
                    names.append(Path(line.split(",", 1)[0]).name)
            if set(names) == ds_set and len(names) == len(ds_names):
                return names
        except Exception:
            continue
    return ds_names

def predict_test_submission():
    image_order = get_image_name_list()
    test_lookup = {}
    for row in ds["test"]:
        test_lookup[Path(row["file_name"]).name] = row
    preds = {}
    for image_name in tqdm(image_order, desc="predict::test"):
        row = test_lookup[image_name]
        cache_path = PRED_DIR / f"{image_name}.json"
        if cache_path.exists():
            try:
                preds[image_name] = json.loads(cache_path.read_text(encoding="utf-8"))
                continue
            except Exception:
                pass
        regions = infer_structured_regions(row["image"], row.get("source", "unknown"))
        regions = strip_confidence(regions)
        cache_path.write_text(json.dumps(regions, ensure_ascii=False), encoding="utf-8")
        preds[image_name] = regions
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
    sub = pd.DataFrame({
        "image": image_order,
        "regions": [json.dumps(preds.get(name, []), ensure_ascii=False) for name in image_order],
    })
    sub_path = SUBMIT_DIR / CFG["submission_name"]
    sub.to_csv(sub_path, index=False)
    return sub_path

def match_pairs(gt_regions, pred_regions, thr=0.5):
    candidates = []
    for i, g in enumerate(gt_regions):
        for j, p in enumerate(pred_regions):
            iou = bbox_iou(g["bbox"], p["bbox"])
            if iou >= thr:
                candidates.append((iou, i, j))
    candidates.sort(reverse=True)
    used_i, used_j, pairs = set(), set(), []
    for iou, i, j in candidates:
        if i in used_i or j in used_j:
            continue
        used_i.add(i); used_j.add(j)
        pairs.append((i, j, iou))
    return pairs, used_i, used_j

def cer(gt_text, pred_text):
    gt = metric_normalize_text(gt_text)
    pr = metric_normalize_text(pred_text)
    if len(gt) == 0:
        return 0.0 if len(pr) == 0 else 1.0
    return Levenshtein.distance(gt, pr) / max(1, len(gt))

def page_text(regions):
    regs = sorted(regions, key=reading_key)
    return " ".join(metric_normalize_text(r.get("text", "")) for r in regs if r.get("type") not in NON_TEXT_TYPES and metric_normalize_text(r.get("text", "")))

def score_page(gt_regions, pred_regions):
    pairs, used_gt, used_pred = match_pairs(gt_regions, pred_regions, thr=0.5)
    tp = len(pairs)
    fp = max(0, len(pred_regions) - tp)
    fn = max(0, len(gt_regions) - tp)
    precision = tp / max(1, tp + fp)
    recall = tp / max(1, tp + fn)
    f1 = 2 * precision * recall / max(1e-9, precision + recall)
    class_acc = np.mean([1.0 if pred_regions[j]["type"] == gt_regions[i]["type"] else 0.0 for i, j, _ in pairs]) if pairs else 0.0
    cers = []
    for i, j, _ in pairs:
        g = gt_regions[i]
        if g.get("language", "uk") == "uk" and g.get("legibility", "legible") == "legible" and g.get("type") not in NON_TEXT_TYPES:
            cers.append(cer(g.get("text", ""), pred_regions[j].get("text", "")))
    region_cer = float(np.mean(cers)) if cers else 1.0
    page_cer = cer(page_text(gt_regions), page_text(pred_regions))
    score = 0.15 * f1 + 0.05 * class_acc + 0.30 * (1 - region_cer) + 0.50 * (1 - page_cer)
    return {
        "score": score,
        "f1": f1,
        "class_acc": class_acc,
        "region_cer": region_cer,
        "page_cer": page_cer,
    }

def run_holdout_eval():
    file_rows = defaultdict(list)
    for row in ds["train"]:
        file_rows[Path(row["file_name"]).name] = row
    val_files = sorted(gt_val_manifest["image_file"].unique().tolist()) if not gt_val_manifest.empty else []
    all_scores = []
    for file_name in tqdm(val_files, desc="eval::holdout"):
        row = file_rows.get(Path(file_name).name)
        if row is None:
            continue
        pred = strip_confidence(infer_structured_regions(row["image"], row.get("source", "unknown")))
        gt_regions = row["regions"] or []
        all_scores.append(score_page(gt_regions, pred))
    if not all_scores:
        return None
    mean_metrics = {k: float(np.mean([x[k] for x in all_scores])) for k in all_scores[0].keys()}
    print(json.dumps(mean_metrics, ensure_ascii=False, indent=2))
    return mean_metrics

if CFG["run_holdout"]:
    _ = run_holdout_eval()

submission_path = None
if CFG["predict_test"]:
    submission_path = predict_test_submission()
    print(str(submission_path))


ModuleNotFoundError: No module named 'surya.texify'